In [ ]:
import csv

input_file = "top_ingredients_flavor_profile_cleaned.csv"
output_file = "top_first_filter.csv"

filtered_rows = []

with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    for row in reader:
        ingredient = row['ingredient_name'].strip().lower()
        count = int(row['count'])
        if ingredient.startswith("for"):
            continue
        filtered_rows.append(row)

# Write filtered data
with open(output_file, mode='w', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=reader.fieldnames)
    writer.writeheader()
    writer.writerows(filtered_rows)

print(f"✅ Saved {len(filtered_rows)} cleaned rows to {output_file}")

✅ Saved 1158 cleaned rows to top_first_filter.csv


In [ ]:
import pandas as pd
import re
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict
from tqdm import tqdm

# Load data
df = pd.read_csv('top_first_filter.csv')

# Load SpaCy model for cleaning
nlp = spacy.load('en_core_web_sm')

# Function to clean ingredient names
def clean_ingredient(ingredient):
    # Regex to remove unnecessary parts (e.g., "to be", "for", etc.)
    cleaned = re.sub(r'\b(to|for|with|into|of|in|the|and|or|a|an|be)\b', '', ingredient)
    cleaned = cleaned.strip().lower()  # Convert to lowercase
    # Use SpaCy to extract only relevant words (i.e., nouns or important terms)
    doc = nlp(cleaned)
    cleaned = ' '.join([token.text for token in doc if token.pos_ in ['NOUN', 'PROPN']])
    return cleaned

# Apply cleaning function to ingredient_name column
df['cleaned_label'] = df['ingredient_name'].apply(clean_ingredient)

# SentenceTransformer model for embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode the cleaned labels
embeddings = model.encode(df['cleaned_label'].tolist(), convert_to_tensor=True)

# Agglomerative Clustering with a distance threshold
clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=1.0)
labels = clustering.fit_predict(embeddings)

# Create a dictionary to store clustered ingredients
clustered_ingredients = defaultdict(list)

# Map each ingredient to its cluster label
for ingredient, label in zip(df['cleaned_label'], labels):
    clustered_ingredients[label].append(ingredient)

# Function to generate the suggested label for each cluster
def generate_suggested_label(clustered_ingredients):
    cluster_suggested_label = {}

    # For each cluster, we will use the most common cleaned label as the representative label
    for label, ingredients in clustered_ingredients.items():
        # Find the most common cleaned label in the cluster
        most_common_label = max(set(ingredients), key=ingredients.count)
        cluster_suggested_label[label] = most_common_label

    return cluster_suggested_label

# Generate suggested labels for each cluster
cluster_suggested_label = generate_suggested_label(clustered_ingredients)

# Map suggested labels to original DataFrame
df['suggested_label'] = [cluster_suggested_label[label] for label in labels]

# Save the updated dataframe with new suggested labels to CSV
df.to_csv('top_ingredients_with_suggested_labels.csv', index=False)

print("\nNew CSV saved with Suggested Labels!")



New CSV saved with Suggested Labels!


In [ ]:
import json
import csv

# Load the JSON (assumed as list of dicts)
with open("IngredientFlavourProfile.json", "r", encoding="utf-8") as json_file:
    ingredient_flavor_list = json.load(json_file)

# Convert to lookup dictionary
flavor_lookup = {
    item["ingredient"].strip().lower(): item["taste profile"]
    for item in ingredient_flavor_list
    if "ingredient" in item and "taste profile" in item
}

input_csv = "top_ingredients_with_suggested_labels.csv"
output_csv = "top_flavor_mapped.csv"

mapped_rows = []

with open(input_csv, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    fieldnames = reader.fieldnames + ["flavour_profile"]

    print(f"📋 Detected columns: {reader.fieldnames}")  # Debug step

    for row in reader:
        label = row.get("suggested_label", "").strip().lower()

        if not label:
            row["flavour_profile"] = "no profile found"
            mapped_rows.append(row)
            continue

        matched = False

        for ingr_key in flavor_lookup:
            if ingr_key in label:
                row["flavour_profile"] = flavor_lookup[ingr_key]
                matched = True
                break

        if not matched:
            row["flavour_profile"] = "no profile found"

        mapped_rows.append(row)

# Save the result
with open(output_csv, mode='w', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(mapped_rows)

print(f"✅ Mapped {len(mapped_rows)} rows with flavor profiles to {output_csv}")


📋 Detected columns: ['ingredient_name', 'count', 'spicy', 'sweet', 'sour', 'bitter', 'salty', 'flavour_profile', 'cleaned_label', 'suggested_label']
✅ Mapped 1158 rows with flavor profiles to top_flavor_mapped.csv


In [ ]:
# Count total rows and how many have "no profile found"
total_rows = len(mapped_rows)
no_profile_count = sum(1 for row in mapped_rows if row["flavour_profile"] == "no profile found")

print(f"🔢 Total rows: {total_rows}")
print(f"❌ Rows with 'no profile found': {no_profile_count}")
print(f"✅ Rows with matched flavor profile: {total_rows - no_profile_count}")


🔢 Total rows: 1158
❌ Rows with 'no profile found': 766
✅ Rows with matched flavor profile: 392


In [ ]:
!pip install fuzzywuzzy
!pip install python-Levenshtein  # Optional for faster performance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from fuzzywuzzy import process

# Load your main dataset with the "tasteprofile" column
main_df = pd.read_csv('top_flavor_mapped.csv')

# Load the ingredient-flavor mapping CSV
flavor_df = pd.read_csv('ingredient_flavor_profiles.csv')

# Define a function to apply fuzzy matching
def get_best_match(ingredient, flavor_df):
    matches = process.extract(ingredient, flavor_df['Ingredient'].str.lower(), limit=1)
    if matches and matches[0][1] >= 80:  # Adjust threshold as needed (higher means stricter match)
        return flavor_df.loc[flavor_df['Ingredient'].str.lower() == matches[0][0], 'Flavor Profile'].values[0]
    return None

# Iterate through the flavor_df and update the tasteprofile where it's "no matched profile"
for _, row in flavor_df.iterrows():
    flavor_ingredient = row['Ingredient'].lower()
    flavor_profile = row['Flavor Profile']

    # Apply match where 'ingredient' contains the flavor ingredient AND tasteprofile is "no matched profile"
    for idx, main_row in main_df[main_df['flavour_profile'] == 'no profile found'].iterrows():
        ingredient_name = main_row['suggested_label']

        # Check if the ingredient name is a valid string before applying lower()
        if isinstance(ingredient_name, str):
            ingredient_name = ingredient_name.lower()
        else:
            continue  # Skip non-string or NaN values

        # Apply fuzzy matching to find the best match
        matched_profile = get_best_match(ingredient_name, flavor_df)

        if matched_profile:
            main_df.at[idx, 'flavour_profile'] = matched_profile

# Save the updated file
main_df.to_csv('new_top_flavour_mapped.csv', index=False)

print(f"✅ Enriched {len(main_df)} rows.")

✅ Enriched 1158 rows.


In [ ]:
# Calculate the total number of rows
total_rows = len(main_df)

# Calculate the total number of "no profile found" rows in the 'flavour_profile' column
no_profile_count = len(main_df[main_df['flavour_profile'] == 'no profile found'])

# Print the results
print(f"🔢 Total rows: {total_rows}")
print(f"❌ Rows with 'no profile found': {no_profile_count}")


🔢 Total rows: 1158
❌ Rows with 'no profile found': 641


In [1]:
!pip install llama-cpp-python \
  --prefer-binary \
  --extra-index-url=https://jllllll.github.io/llama-cpp-python-cuBLAS-wheels/AVX2/cu121

Looking in indexes: https://pypi.org/simple, https://jllllll.github.io/llama-cpp-python-cuBLAS-wheels/AVX2/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00


In [2]:
!wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf

--2025-04-26 12:55:13--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.34, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs.hf.co/repos/46/12/46124cd8d4788fd8e0879883abfc473f247664b987955cc98a08658f7df6b826/14466f9d658bf4a79f96c3f3f22759707c291cac4e62fea625e80c7d32169991?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27mistral-7b-instruct-v0.1.Q4_K_M.gguf%3B+filename%3D%22mistral-7b-instruct-v0.1.Q4_K_M.gguf%22%3B&Expires=1745674745&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0NTY3NDc0NX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy80Ni8xMi80NjEyNGNkOGQ0Nzg4ZmQ4ZTA4Nzk4ODNhYmZjNDczZjI0NzY2NGI5ODc5NTVjYzk4YTA4NjU4ZjdkZjZiODI2LzE0NDY2ZjlkNjU4YmY0YTc5Zjk2YzNmM2YyMjc1O

In [2]:
import pandas as pd
from llama_cpp import Llama

# Load your local Mistral model
llm = Llama(
    model_path="mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=1500,  # Context window
    n_gpu_layers=40,  # Try increasing layers on GPU
    n_threads=8,  # Using all available cores
    n_batch=16
)

# Load your CSV
df = pd.read_csv("new_top_flavour_mapped.csv")

# Flavor prediction prompt template
def get_prompt(ingredient):
    return f"""You're a food scientist. Tell me what are the basic tastes of the ingredient: "{ingredient}".
Choose from: sweet, salty, sour, bitter, spicy, umami, astringent.
Return only the relevant taste(s) as a comma-separated list."""

# Generate flavor for a single ingredient
def predict_flavor(ingredient):
    # Edge case check
    if not isinstance(ingredient, str) or "ingredient" in ingredient.lower() or "deepfrying" in ingredient.lower():
        return "unknown"

    prompt = get_prompt(ingredient)

    try:
        output = llm(
            prompt=prompt,
            max_tokens=100,
            temperature=0.5,
            stop=None
        )
        response = output["choices"][0]["text"].strip().lower()
        return response
    except Exception as e:
        print(f"Error for '{ingredient}': {e}")
        return "unknown"

# Generate for "No profile found"
suggested_flavors = []
for _, row in df.iterrows():
    if str(row['flavour_profile']).lower().strip() == "no profile found":
        predicted = predict_flavor(row['ingredient_name'])
        suggested_flavors.append(predicted)
        print(f"Predicted flavor for '{row['ingredient_name']}': {predicted}")  # Print statement added
    else:
        suggested_flavors.append("")

# Add the new column
df['suggested_flavor'] = suggested_flavors

# Save it
df.to_csv("ingredient_flavors_updated.csv", index=False)
print("✅ Done! Output saved to 'ingredient_flavors_updated.csv'")


AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


Predicted flavor for 'oil': the basic tastes of oil are: salty, umami, astringent.


Llama.generate: prefix-match hit


Predicted flavor for 'asafoetida hing': 


Llama.generate: prefix-match hit


...

Predicted flavor for 'yam suran cut into mm': the yam suran is a type of japanese noodle made with tapioca flour and mashed yam. the taste of yam suran can be described as sweet and umami, while salty and bitter flavors are not typically present. spicy and sour flavors are also not typical of yam suran, but they may be added to the dish through seasonings or sauces.


Llama.generate: prefix-match hit


Predicted flavor for 'yellow moong dal split yellow soaked for hours and drained': "sweet, salty, sour, bitter, spicy, umami, astringent."


Llama.generate: prefix-match hit


Predicted flavor for 'yellow moong dal split yellow soaked for minutes and drained': the basic tastes of yellow moong dal are: sweet, bitter, and astringent.
✅ Done! Output saved to 'ingredient_flavors_updated.csv'


In [6]:
import pandas as pd
import re

# Load the ingredient flavor dataset
df = pd.read_csv("ingredient_flavors_updated.csv")

# Load your master flavor list from Excel
flavors_df = pd.read_excel("Scaled_Flavour_Profile.xlsx")  # <-- your Excel file
all_flavors = flavors_df['Flavour'].dropna().str.lower().str.strip().unique().tolist()

# Compile regex pattern to match any of the allowed flavors
flavor_pattern = r'\b(?:' + '|'.join(map(re.escape, all_flavors)) + r')\b'

# Define the cleaning function
def extract_flavors(text):
    if pd.isna(text):
        return "unknown"

    text = str(text).lower()

    # If model output contains an apology or unclear instruction, treat as unknown
    if "i'm sorry" in text or "please provide more details" in text or "for example" in text:
        return "unknown"

    # Find matching flavors from your master list
    matched_flavors = re.findall(flavor_pattern, text)

    if matched_flavors:
        return ", ".join(sorted(set(matched_flavors)))
    else:
        return "unknown"

# Clean the 'suggested_flavor' column first
df['cleaned_suggested_flavor'] = df['suggested_flavor'].apply(extract_flavors)

# Replace 'flavor_profile' values where missing
df['flavour_profile'] = df.apply(
    lambda row: row['cleaned_suggested_flavor'] if str(row['flavour_profile']).lower().strip() == "no profile found" else row['flavour_profile'],
    axis=1
)

# Drop the helper columns
df.drop(columns=['suggested_flavor', 'cleaned_suggested_flavor'], inplace=True)

# Save the updated DataFrame
df.to_csv("ingredient_flavors_updated.csv", index=False)
print("✅ Flavor profiles updated with master flavor list and saved to 'ingredient_flavors_updated.csv'")


✅ Flavor profiles updated with master flavor list and saved to 'ingredient_flavors_updated.csv'


In [11]:
import pandas as pd
import re

# Load main and mapping files
main_df = pd.read_csv("ingredient_flavors_updated.csv")
mapping_df = pd.read_excel("Scaled_Flavour_Profile.xlsx")  # your Excel file

# Clean mapping columns
mapping_df.columns = mapping_df.columns.str.strip().str.lower()
mapping_df = mapping_df.rename(columns={
    "flavour": "flavour",
    "sourness": "sour",
    "sweetness": "sweet",
    "saltyness": "salty",
    "spicy level": "spicy",
    "bitterness": "bitter",
    "umaminess": "umami"  # ✅ Add this line if column is named like this in your excel
})

# Ensure numeric types
for col in ['sour', 'sweet', 'salty', 'spicy', 'bitter', 'umami']:   # ✅ Include 'umami'
    mapping_df[col] = pd.to_numeric(mapping_df[col], errors='coerce').fillna(0).astype(int)

# Create mapping dict
flavour_map = mapping_df.set_index('flavour').T.to_dict()
known_flavours = list(flavour_map.keys())

# Add taste columns
for taste in ['sour', 'sweet', 'salty', 'spicy', 'bitter', 'umami']:  # ✅ Add 'umami'
    main_df[taste] = 0

# Main loop with regex-based splitting
for idx, row in main_df.iterrows():
    profile = row['flavour_profile']
    if isinstance(profile, str) and profile.strip().lower() != "unknown":
        taste_totals = {k: 0 for k in ['sour', 'sweet', 'salty', 'spicy', 'bitter', 'umami']}  # ✅ Add 'umami'

        # Split on anything not a letter or number
        flavour_tokens = re.split(r'[^a-zA-Z0-9]+', profile.lower())

        for token in flavour_tokens:
            token = token.strip()
            if not token:
                continue
            for known in known_flavours:
                if known == token:
                    for taste in taste_totals:
                        taste_totals[taste] += flavour_map[known].get(taste, 0)
                    break  # avoid matching same token multiple times

        for taste, val in taste_totals.items():
            main_df.at[idx, taste] = val

# Save result
main_df.to_csv("flavour_scaled_cleaned.csv", index=False)
print("✅ Flavour scaling (including umami) complete — file saved as 'flavour_scaled_cleaned.csv'")


✅ Flavour scaling (including umami) complete — file saved as 'flavour_scaled_cleaned.csv'


In [12]:
# Calculate the total number of rows
import pandas as pd
df = pd.read_csv("flavour_scaled_cleaned.csv")
total_rows = len(df)

# Calculate the total number of "no profile found" rows in the 'flavour_profile' column
no_profile_count = len(df[df['flavour_profile'] == 'unknown'])

# Print the results
print(f"🔢 Total rows: {total_rows}")
print(f"❌ Rows with 'no profile found': {no_profile_count}")

🔢 Total rows: 1158
❌ Rows with 'no profile found': 293
